# Calentamiento antes de descongelar

Dos de las nueve arquitecturas colapsaban al azar cuando el ajuste fino empezaba contra una cabeza aleatoria: `descongelarCola` deja las `BatchNormalization` en las estadísticas de ImageNet mientras las convoluciones ya se mueven. Aquí se mide si basta con calentar, con la receta real de los nueve notebooks —misma tasa y mismos pesos de clase en ambas fases—, de modo que la única variable sea la fase congelada.


In [ ]:
# En Kaggle clona el repo; en local no hace nada.
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/CCSandoval/ButterflyModeling.git"

if Path("/kaggle").exists():
    CLON = "/tmp/ButterflyModeling"
    if not Path(CLON).exists():                     # el repo es público: clonar no pide credenciales
        subprocess.run(["git", "clone", "--depth", "1", REPO, CLON], check=True)
    sys.path.insert(0, CLON)
    os.chdir(CLON)
    print("clon en", CLON, "·", subprocess.run(
        ["git", "log", "-1", "--format=%h %s"], cwd=CLON,
        capture_output=True, text=True).stdout.strip())

    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        print("sin GITHUB_TOKEN: no se publicará")

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "corpus.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Raíz del proyecto:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from scripts import corpus
from scripts import modelRegistry as registro
from scripts.architectures import IMG_SIZE, buildModel, descongelarCola, preprocessFn
from scripts.dataPipeline import aumentadorFotometrico, loadSplit, preparar
from scripts.vizStyle import SERIE_1, SERIE_2, applyStyle

ARQUITECTURAS = ("mobilenet_v2", "resnet50_v2", "mobilenet_v3_small")

LOTE = 32
EPOCAS_CALENTAMIENTO = 5
EPOCAS_AJUSTE = 8
TASA = 1e-4              # la misma de los nueve notebooks, en las dos fases
SEMILLA = 42

FIGURAS = registro.OUTPUTS_DIR / "figuras"
FIGURAS.mkdir(exist_ok=True)

tf.keras.utils.set_random_seed(SEMILLA)
corpus.materializar()

dsTrainCrudo, clases = loadSplit("dataset/train", IMG_SIZE, LOTE)
dsValCrudo, _ = loadSplit("dataset/validate", IMG_SIZE, LOTE, shuffle=False)
pesosClase = corpus.classWeights()
NUM_CLASES = len(clases)
AZAR = 1.0 / NUM_CLASES
print(f"{NUM_CLASES} clases · azar {AZAR:.4f} · {IMG_SIZE[0]}x{IMG_SIZE[1]}")

## 1 · Con y sin calentamiento

`calentar=False` reproduce la configuración que colapsaba.

In [ ]:
def entrenarEnDosFases(arquitectura, calentar):
    tf.keras.utils.set_random_seed(SEMILLA)
    preprocess = preprocessFn(arquitectura)
    modelo = buildModel(arquitectura, NUM_CLASES, aumentar=True)
    dsT = preparar(dsTrainCrudo, preprocess, aumentadorFotometrico())
    dsV = preparar(dsValCrudo, preprocess)
    historia = {}

    if calentar:
        modelo.compile(optimizer=tf.keras.optimizers.Adam(TASA),
                       loss="categorical_crossentropy", metrics=["accuracy"])
        historia = modelo.fit(dsT, validation_data=dsV, epochs=EPOCAS_CALENTAMIENTO,
                              class_weight=pesosClase, verbose=0).history

    descongelarCola(modelo, 0.30)
    modelo.compile(optimizer=tf.keras.optimizers.Adam(TASA),
                   loss="categorical_crossentropy", metrics=["accuracy"])
    ajuste = modelo.fit(dsT, validation_data=dsV, epochs=EPOCAS_AJUSTE,
                        class_weight=pesosClase, verbose=0).history

    del modelo
    tf.keras.backend.clear_session()
    return {k: historia.get(k, []) + v for k, v in ajuste.items()}


curvas, fases = {}, []
for arquitectura in ARQUITECTURAS:
    for calentar in (False, True):
        h = entrenarEnDosFases(arquitectura, calentar)
        curvas[(arquitectura, calentar)] = h
        fases.append({
            "arquitectura": arquitectura,
            "calentamiento": calentar,
            "train_acc": h["accuracy"][-1],
            "val_acc": h["val_accuracy"][-1],
            "val_loss_max": max(h["val_loss"]),
            "colapsa": h["accuracy"][-1] < 5 * AZAR,
        })
        print(f"{arquitectura:20s} calentamiento={str(calentar):5s} "
              f"train={h['accuracy'][-1]:.4f} val={h['val_accuracy'][-1]:.4f} "
              f"pico_val_loss={max(h['val_loss']):9.2f}")

resultados = pd.DataFrame(fases)
resultados

## 2 · Las curvas

El valor de la última época no distingue «aprendió despacio» de «nunca despegó».

In [ ]:
applyStyle()
import matplotlib.pyplot as plt

fig, ejes = plt.subplots(1, len(ARQUITECTURAS), figsize=(4 * len(ARQUITECTURAS), 3.6),
                         sharey=True)
for eje, arquitectura in zip(ejes, ARQUITECTURAS):
    for calentar, color in ((False, SERIE_2), (True, SERIE_1)):
        serie = curvas[(arquitectura, calentar)]["val_accuracy"]
        eje.plot(range(1, len(serie) + 1), serie, color=color, linewidth=1.8,
                 label="con calentamiento" if calentar else "sin calentamiento")
    eje.axhline(AZAR, color="#8a8880", linestyle="--", linewidth=1.1)
    eje.axvline(EPOCAS_CALENTAMIENTO + 0.5, color="#c9c6bd", linewidth=1)
    eje.set(title=arquitectura, xlabel="época")
ejes[0].set_ylabel("accuracy (validación)")
ejes[0].text(0.3, AZAR, " azar", fontsize=8, va="bottom", color="#8a8880")
ejes[-1].legend(loc="lower right", frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIGURAS / "calentamiento.png", dpi=300, bbox_inches="tight")
plt.show()

## Guardar resultados

In [ ]:
for arquitectura in ARQUITECTURAS:
    sub = resultados[resultados.arquitectura == arquitectura].set_index("calentamiento")
    sin, con = sub.loc[False], sub.loc[True]
    veredicto = ("rescatada" if sin["colapsa"] and not con["colapsa"]
                 else "colapsa igual" if sin["colapsa"]
                 else "ya funcionaba")
    print(f"{arquitectura:20s} sin {sin['val_acc']:.3f} -> con {con['val_acc']:.3f} "
          f"({con['val_acc'] - sin['val_acc']:+.3f})  {veredicto}")

registro.writeJson(registro.OUTPUTS_DIR / "calentamiento.json", {
    "num_clases": NUM_CLASES,
    "azar": round(AZAR, 4),
    "epocas_calentamiento": EPOCAS_CALENTAMIENTO,
    "epocas_ajuste": EPOCAS_AJUSTE,
    "tasa": TASA,
    "resultados": resultados.to_dict("records"),
})

from scripts.kaggle import publicarArchivos
if Path("/kaggle").exists() and os.environ.get("GITHUB_TOKEN"):
    publicarArchivos("Add warm-up diagnostic",
                     "outputs/calentamiento.json",
                     "outputs/figuras/calentamiento.png")
